# 🏥 Ophthalmology Multi-Dataset Harmonization

This notebook builds a reproducible, scalable harmonization pipeline across a large set of ophthalmology datasets from Kaggle.

## Goals:
- Load datasets with inconsistent formats
- Standardize metadata into a unified schema
- Create reproducible harmonization rules
- Export a final combined dataset for ML tasks

This notebook favors clarity and teaching. Every function is documented.

In [ ]:
import os
import re
import json
import pandas as pd
import numpy as np
from pathlib import Path
import logging
import warnings

# Suppress common warnings for cleaner output
warnings.filterwarnings('ignore', category=UserWarning)

# Configure pandas display
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.width", 120)

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Create output directory if it doesn't exist
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

print("✓ Libraries imported successfully")
print("✓ Environment configured for data processing")

✓ Libraries imported successfully


In [ ]:
# --- Kaggle authentication + required packages ---
import shutil
import subprocess
import sys
from pathlib import Path

def ensure_packages(packages):
    missing = []
    for pkg in packages:
        try:
            __import__(pkg)
        except ImportError:
            missing.append(pkg)
    if missing:
        print(f"Installing missing packages: {missing}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing, "-q"])


def setup_kaggle_auth():
    """Copy project kaggle.json to user config dir and set env for Kaggle tools."""
    project_kaggle = Path("..") / "kaggle" / "kaggle.json"
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    dest = kaggle_dir / "kaggle.json"

    if project_kaggle.exists():
        shutil.copy2(project_kaggle, dest)
        if os.name != "nt":
            os.chmod(dest, 0o600)
        os.environ["KAGGLE_CONFIG_DIR"] = str(kaggle_dir)
        return True, dest
    return False, project_kaggle

# Ensure packages used for real data and outputs
ensure_packages(["kagglehub", "pyarrow"])

# Configure Kaggle auth
ok, kaggle_path = setup_kaggle_auth()
if ok:
    print(f"✓ Kaggle credentials ready at {kaggle_path}")
else:
    print(f"✗ kaggle.json not found at {kaggle_path}")


## Canonical Harmonization Schema

We standardize all datasets into a unified structure that captures:
- **Required fields**: Core identifiers and classifications
- **Optional fields**: Metadata extracted when available
- **Extra JSON**: Non-standard fields stored as JSON for extensibility

## Schema Design Principles
- **Required Fields**: Core identifiers and classifications present in all records
- **Optional Fields**: Metadata extracted when available from source datasets
- **Extensibility**: Non-standard fields stored as JSON for future compatibility
- **Type Safety**: Clear data types and validation rules

## Field Categories
🏷️ **Core Identifiers** | 👁️ **Image Characteristics** | 🏥 **Medical Data** | 👤 **Patient Metadata** | 📐 **Technical Specs**

In [5]:
# Canonical schema definition
CANONICAL_COLUMNS = [
    # Core identifiers
    "image_id",                    # Unique identifier per image
    "dataset_name",                # Source dataset name
    "image_path",                  # Path or filename of the image
    
    # Image characteristics
    "eye",                         # 'left', 'right', or None
    "modality",                    # 'Fundus', 'OCT', 'Slit-Lamp', etc.
    "view_type",                   # 'macula', 'optic_disc', 'full_field', None
    
    # Diagnosis information
    "diagnosis_raw",               # Original diagnosis from dataset
    "diagnosis_category",          # Normalized diagnosis (DR, AMD, etc.)
    "diagnosis_binary",            # 'Normal' vs 'Abnormal' classification
    "severity",                    # Severity grading if available
    
    # Patient metadata
    "patient_id",                  # De-identified patient identifier
    "age",                         # Patient age in years
    "sex",                         # 'M', 'F', or None
    
    # Image metadata
    "resolution_x",                # Horizontal resolution in pixels
    "resolution_y",                # Vertical resolution in pixels
    
    # Extensibility
    "extra_json"                   # JSON-encoded non-standard fields
]

def canonical_row():
    """Return an empty row matching the canonical schema."""
    return {col: None for col in CANONICAL_COLUMNS}

print(f"✓ Canonical schema defined with {len(CANONICAL_COLUMNS)} columns")
print(f"  Columns: {CANONICAL_COLUMNS}")

✓ Canonical schema defined with 16 columns
  Columns: ['image_id', 'dataset_name', 'image_path', 'eye', 'modality', 'view_type', 'diagnosis_raw', 'diagnosis_category', 'diagnosis_binary', 'severity', 'patient_id', 'age', 'sex', 'resolution_x', 'resolution_y', 'extra_json']


## Basic Harmonization Rules

These rules standardize diagnoses, infer metadata, and normalize terminology.

In [ ]:
# Diagnosis mapping: raw labels → standardized categories
DIAGNOSIS_MAPPING = {
    '0': 'Normal',
    '1': 'DR',
    '2': 'DR',
    '3': 'DR',
    '4': 'DR',
    'no dr': 'Normal',
    'dr': 'DR',
    'diabetic retinopathy': 'DR',
    'retinopathy': 'DR',
    'mild': 'DR',
    'moderate': 'DR',
    'severe': 'DR',
    'proliferative': 'DR',
    'amd': 'AMD',
    'age-related macular degeneration': 'AMD',
    'macular degeneration': 'AMD',
    'cnv': 'AMD',
    'choroidal neovascularization': 'AMD',
    'drusen': 'AMD',
    'cataract': 'Cataract',
    'glaucoma': 'Glaucoma',
    'normal': 'Normal',
    'healthy': 'Normal',
    'fluid': 'Edema',
    'cyst': 'Edema',
    'edema': 'Edema',
    'dme': 'Edema',
    'diabetic macular edema': 'Edema',
    'cornea': 'Corneal Disease',
    'retinoblastoma': 'Retinoblastoma',
    'myopia': 'Myopia',
}

def map_diagnosis(raw):
    """Normalize raw diagnosis label to standardized category."""
    if raw is None:
        return None
    
    r = str(raw).lower().strip()
    
    # Direct lookup
    if r in DIAGNOSIS_MAPPING:
        return DIAGNOSIS_MAPPING[r]
    
    # Substring matching
    for key, normalized in DIAGNOSIS_MAPPING.items():
        if key in r:
            return normalized
    
    return 'Other'

def diagnose_binary(diagnosis_category):
    """Convert diagnosis category to binary: Normal vs Abnormal."""
    if diagnosis_category is None:
        return None
    if diagnosis_category == 'Normal':
        return 'Normal'
    return 'Abnormal'

def infer_eye(path):
    """Infer eye (left/right) from image path or filename."""
    if not isinstance(path, str):
        return None
    
    p = path.lower()
    
    # Left eye patterns
    if any(x in p for x in ['left', '_l', '-l', ' os', '_os', 'left_', 'l.jp', 'left.', 'left-']):
        return 'left'
    
    # Right eye patterns
    if any(x in p for x in ['right', '_r', '-r', ' od', '_od', 'right_', 'r.jp', 'right.', 'right-']):
        return 'right'
    
    return None

def infer_modality(dataset_name):
    """Infer imaging modality from dataset name."""
    name = dataset_name.lower()
    
    if 'oct' in name:
        return 'OCT'
    if 'fundus' in name or 'messidor' in name or 'aptos' in name or 'dr detection' in name:
        return 'Fundus'
    if 'cataract' in name:
        return 'Slit-Lamp'
    if 'cornea' in name:
        return 'Slit-Lamp'
    if 'retinoblastoma' in name:
        return 'Fundus'
    if 'macular' in name or 'amd' in name:
        return 'Fundus'
    
    return 'Unknown'

print("✓ Harmonization rules defined")
print(f"  Diagnosis categories: {len(DIAGNOSIS_MAPPING)}")
print(f"  Sample mappings: {dict(list(DIAGNOSIS_MAPPING.items())[:5])}")


✓ Harmonization rules defined
  Diagnosis categories: 15
  Sample mappings: {'dr': 'DR', 'diabetic retinopathy': 'DR', 'retinopathy': 'DR', 'amd': 'AMD', 'age-related macular degeneration': 'AMD'}


## Universal Loader

This function provides a single interface for loading heterogeneous datasets:
1. Auto-detects image and diagnosis columns
2. Converts rows into canonical format
3. Stores unmapped fields in `extra_json`

In [ ]:
from src.loaders import UniversalLoader, build_dataset_registry, load_and_harmonize_inputs

print("✓ Loader registry helpers imported")

✓ Universal loader defined


In [ ]:
# Auto-detect datasets from INPUT
INPUT_ROOT = Path("..") / "INPUT"
registry_overrides = {}  # Optional per-dataset overrides
dataset_registry = build_dataset_registry(INPUT_ROOT, registry_overrides)

print(f"✓ Auto-detected {len(dataset_registry)} datasets in {INPUT_ROOT}")
for config in dataset_registry[:10]:
    print(f"  - {config.name} (enabled={config.enabled}, data_dir={config.data_dir})")
if len(dataset_registry) > 10:
    print(f"  ...and {len(dataset_registry) - 10} more")

## Dataset Registry

Datasets are auto-detected from INPUT. You can disable or override column mappings using the registry overrides map in the previous cell.

In [ ]:
# Optional: curated Kaggle list for downloads (independent of INPUT auto-detection)
KAGGLE_DATASETS = [
    # (kaggle_identifier, display_name, enabled)
    ("sheemazain/cataract-classification-dataset-in-ds", "Cataract DS", True),
    ("drbasanthkb/cornea-in-diabetes", "Cornea in Diabetes", True),
    ("pritpal2873/diabetic-retinopathy-detection-classification-data", "DR Detection", True),
    ("sumit17125/eye-image-dataset", "Eye Image Dataset", True),
    ("arjunbhushan005/fundus-images", "Fundus Images", True),
    ("orvile/macular-degeneration-disease-dataset", "Macular Degeneration", True),
    ("google-brain/messidor2-dr-grades", "Messidor2", True),
    ("orvile/octdl-optical-coherence-tomography-dataset", "OCTDL", True),
    ("shakilrana/octdl-retinal-oct-images-dataset", "OCTDL Images", True),
    ("ferencjuhsz/refuge2-and-refuge2cross-dataset", "Refuge2", True),
    ("mohamedabdalkader/retinal-disease-detection", "Retinal Disease Detection", True),
    ("andrewmvd/retinal-disease-classification", "Retinal Disease Classification", True),
    ("shuvokumarbasak2030/retinal-colorized-oct-images", "Retinal Colorized OCT", True),
]

KAGGLE_DOWNLOADS_DIR = Path("..") / "INPUT" / "kaggle_downloads"

print(f"✓ Kaggle registry loaded with {len(KAGGLE_DATASETS)} datasets")
print(f"  Enabled: {sum(1 for _, _, e in KAGGLE_DATASETS if e)}")
print(f"  Kaggle downloads: {KAGGLE_DOWNLOADS_DIR}")

✓ Dataset registry loaded with 12 datasets
  Enabled: 12


## Integrate Real Kaggle Data
Use Kaggle API instead of demo data. Prereqs: install the `kaggle` CLI (`pip install kaggle`), set `KAGGLE_USERNAME` and `KAGGLE_KEY` env vars, and ensure internet access. The cell below downloads the enabled datasets from `KAGGLE_DATASETS`, unzips them into `INPUT/kaggle_downloads/<dataset>/`, and shows a quick file listing.

In [ ]:
# Download enabled Kaggle datasets into INPUT/kaggle_downloads
import kagglehub
from pathlib import Path

KAGGLE_DOWNLOADS_DIR.mkdir(parents=True, exist_ok=True)

def download_kaggle_dataset(kaggle_id, display_name):
    try:
        logger.info(f"Downloading {kaggle_id}...")
        dataset_path = Path(kagglehub.dataset_download(kaggle_id))
        target_dir = KAGGLE_DOWNLOADS_DIR / display_name.replace(" ", "_")
        target_dir.mkdir(parents=True, exist_ok=True)

        # Copy cached download into project INPUT folder for harmonization
        if dataset_path.exists():
            for item in dataset_path.iterdir():
                dest = target_dir / item.name
                if item.is_dir():
                    if not dest.exists():
                        shutil.copytree(item, dest)
                else:
                    if not dest.exists():
                        shutil.copy2(item, dest)
        logger.info(f"✓ {display_name} ready at {target_dir}")
        return target_dir
    except Exception as e:
        logger.warning(f"✗ Failed to download {display_name}: {e}")
        return None

kaggle_downloaded_dirs = []
for kaggle_id, display_name, enabled in KAGGLE_DATASETS:
    if not enabled:
        continue
    out_dir = download_kaggle_dataset(kaggle_id, display_name)
    if out_dir:
        kaggle_downloaded_dirs.append(out_dir)

print(f"✓ Kaggle downloads completed: {len(kaggle_downloaded_dirs)} datasets")

## Auto-Detect or Create Demo Datasets

The loader now auto-detects datasets from INPUT. If none are found, we generate realistic demo datasets for a complete end-to-end run.

In [ ]:
# Auto-detect and harmonize datasets from INPUT
harmonized_frames, load_results, failed_datasets = load_and_harmonize_inputs(INPUT_ROOT, registry_overrides)
input_datasets = [(r.name, r.dataframe) for r in load_results]

print(f"✓ Harmonized {len(harmonized_frames)} datasets from INPUT")
if failed_datasets:
    print(f"⚠️ Failed datasets: {failed_datasets}")

# Fallback demo datasets if none are available
if not input_datasets:
    demo_datasets = {}

    # 1. Cataract Dataset
    demo_datasets['Cataract DS'] = pd.DataFrame({
        'image_path': ['cat_001_right.jpg', 'cat_001_left.jpg', 'cat_002_right.jpg', 'cat_002_left.jpg'],
        'condition': ['Immature Cataract', 'Healthy', 'Mature Cataract', 'Healthy'],
        'age': [67, 67, 71, 71],
        'sex': ['M', 'M', 'F', 'F']
    })

    # 2. Cornea Dataset
    demo_datasets['Cornea in Diabetes'] = pd.DataFrame({
        'filename': ['cornea_001_od.png', 'cornea_001_os.png', 'cornea_002_od.png'],
        'label': ['Healthy', 'Corneal Damage', 'Healthy'],
        'patient_age': [45, 45, 58]
    })

    # 3. DR Detection Dataset
    demo_datasets['DR Detection'] = pd.DataFrame({
        'id_code': ['10005_right', '10005_left', '10007_right', '10007_left', '10009_right'],
        'diagnosis': [2, 0, 1, 1, 4],
        'path': ['10005_right.png', '10005_left.png', '10007_right.png', '10007_left.png', '10009_right.png']
    })

    # 4. OCT Dataset
    demo_datasets['OCTDL'] = pd.DataFrame({
        'scan_id': ['OCT_001', 'OCT_002', 'OCT_003', 'OCT_004'],
        'label': ['Normal', 'AMD', 'Normal', 'DME'],
        'resolution_x': [512, 512, 512, 512],
        'resolution_y': [496, 496, 496, 496]
    })

    # 5. Fundus Images Dataset
    demo_datasets['Fundus Images'] = pd.DataFrame({
        'image_name': ['fundus_001.jpg', 'fundus_002.jpg', 'fundus_003.jpg'],
        'disease': ['Diabetic Retinopathy', 'Normal', 'Diabetic Retinopathy'],
        'age_years': [52, 45, 67]
    })

    input_datasets = list(demo_datasets.items())
    harmonized_frames = []
    load_results = []
    failed_datasets = []

    print("✓ Demo datasets created:")
    for name, df in demo_datasets.items():
        print(f"  {name}: {len(df)} records, columns={list(df.columns)}")
        loader = UniversalLoader(name)
        harmonized_df = loader.load_and_harmonize(df)
        if not harmonized_df.empty:
            harmonized_frames.append(harmonized_df)
            load_results.append((name, harmonized_df))
        else:
            failed_datasets.append(name)
else:
    for name, df in input_datasets:
        print(f"  {name}: {len(df)} records, columns={list(df.columns)[:8]}")

✓ Demo datasets created:
  Cataract DS: 4 records, columns=['image_path', 'condition', 'age', 'sex']
  Cornea in Diabetes: 3 records, columns=['filename', 'label', 'patient_age']
  DR Detection: 5 records, columns=['id_code', 'diagnosis', 'path']
  OCTDL: 4 records, columns=['scan_id', 'label', 'resolution_x', 'resolution_y']
  Fundus Images: 3 records, columns=['image_name', 'disease', 'age_years']


## Harmonization Pipeline

Load, harmonize, merge, and export all datasets.

In [ ]:
# Harmonization summary
if not harmonized_frames:
    print("✗ No datasets were harmonized. Check INPUT or demo datasets.")
else:
    print(f"\n{'='*60}")
    print(f"Processed {len(harmonized_frames)} datasets successfully")
    if failed_datasets:
        print(f"Failed datasets: {failed_datasets}")
    if load_results:
        print("\nDataset reports (first 5):")
        for result in load_results[:5]:
            if hasattr(result, "report"):
                print(f"- {result.name}: errors={result.report.get('total_errors')}, warnings={result.report.get('total_warnings')}")
            else:
                name = result[0] if isinstance(result, tuple) else "unknown"
                print(f"- {name}")

2025-12-06 19:40:31,528 - INFO - Loading dataset: Cataract DS
2025-12-06 19:40:31,530 - INFO -   Auto-detected columns: img=image_path, diag=condition, eye=None
2025-12-06 19:40:31,539 - INFO -   Harmonized 4 records from Cataract DS



Processing: Cataract DS
  Original shape: (4, 4)
  Original columns: ['image_path', 'condition', 'age', 'sex']


2025-12-06 19:40:31,541 - INFO - Loading dataset: Cornea in Diabetes
2025-12-06 19:40:31,542 - INFO -   Auto-detected columns: img=filename, diag=label, eye=None
2025-12-06 19:40:31,545 - INFO -   Harmonized 3 records from Cornea in Diabetes
2025-12-06 19:40:31,546 - INFO - Loading dataset: DR Detection
2025-12-06 19:40:31,547 - INFO -   Auto-detected columns: img=path, diag=diagnosis, eye=id_code
2025-12-06 19:40:31,550 - INFO -   Harmonized 5 records from DR Detection
2025-12-06 19:40:31,551 - INFO - Loading dataset: OCTDL
2025-12-06 19:40:31,552 - INFO -   Auto-detected columns: img=None, diag=label, eye=None
2025-12-06 19:40:31,555 - INFO -   Harmonized 4 records from OCTDL
2025-12-06 19:40:31,556 - INFO - Loading dataset: Fundus Images
2025-12-06 19:40:31,557 - INFO -   Auto-detected columns: img=image_name, diag=disease, eye=None
2025-12-06 19:40:31,560 - INFO -   Harmonized 3 records from Fundus Images


  ✓ Harmonized shape: (4, 16)

Processing: Cornea in Diabetes
  Original shape: (3, 3)
  Original columns: ['filename', 'label', 'patient_age']
  ✓ Harmonized shape: (3, 16)

Processing: DR Detection
  Original shape: (5, 3)
  Original columns: ['id_code', 'diagnosis', 'path']
  ✓ Harmonized shape: (5, 16)

Processing: OCTDL
  Original shape: (4, 4)
  Original columns: ['scan_id', 'label', 'resolution_x', 'resolution_y']
  ✓ Harmonized shape: (4, 16)

Processing: Fundus Images
  Original shape: (3, 3)
  Original columns: ['image_name', 'disease', 'age_years']
  ✓ Harmonized shape: (3, 16)

Processed 5 datasets successfully


## Merge All Datasets

In [12]:
# Merge all harmonized dataframes
if harmonized_frames:
    final_df = pd.concat(harmonized_frames, ignore_index=True)
    print(f"✓ Merged dataset created")
    print(f"  Total records: {len(final_df)}")
    print(f"  Columns: {len(final_df.columns)}")
    print(f"\n  Shape: {final_df.shape}")
else:
    print("✗ No datasets to merge")
    final_df = pd.DataFrame(columns=CANONICAL_COLUMNS)

✓ Merged dataset created
  Total records: 19
  Columns: 16

  Shape: (19, 16)


## Data Exploration and Quality Checks

In [13]:
# Display sample records
print("\n=== SAMPLE HARMONIZED RECORDS ===")
print(final_df.head(10).to_string())


=== SAMPLE HARMONIZED RECORDS ===
               image_id        dataset_name         image_path    eye   modality view_type      diagnosis_raw diagnosis_category diagnosis_binary severity patient_id   age   sex resolution_x resolution_y extra_json
0         Cataract DS_0         Cataract DS  cat_001_right.jpg  right  Slit-Lamp      None  Immature Cataract           Cataract         Abnormal     None       None    67     M         None         None       None
1         Cataract DS_1         Cataract DS   cat_001_left.jpg   left  Slit-Lamp      None            Healthy             Normal           Normal     None       None    67     M         None         None       None
2         Cataract DS_2         Cataract DS  cat_002_right.jpg  right  Slit-Lamp      None    Mature Cataract           Cataract         Abnormal     None       None    71     F         None         None       None
3         Cataract DS_3         Cataract DS   cat_002_left.jpg   left  Slit-Lamp      None            Hea

## Dataset Statistics

In [14]:
# Column-wise statistics
print("\n=== DATASET STATISTICS ===")
print(f"Total records: {len(final_df)}")
print(f"Total datasets: {final_df['dataset_name'].nunique()}")
print(f"\nRecords per dataset:")
print(final_df['dataset_name'].value_counts().sort_index())


=== DATASET STATISTICS ===
Total records: 19
Total datasets: 5

Records per dataset:
dataset_name
Cataract DS           4
Cornea in Diabetes    3
DR Detection          5
Fundus Images         3
OCTDL                 4
Name: count, dtype: int64


In [15]:
# Diagnosis distribution
print("\n=== DIAGNOSIS DISTRIBUTION ===")
print("\nNormalized diagnoses:")
print(final_df['diagnosis_category'].value_counts(dropna=False))

print("\nBinary classification:")
print(final_df['diagnosis_binary'].value_counts(dropna=False))


=== DIAGNOSIS DISTRIBUTION ===

Normalized diagnoses:
diagnosis_category
Normal             7
Other              6
Cataract           2
DR                 2
Corneal Disease    1
AMD                1
Name: count, dtype: int64

Binary classification:
diagnosis_binary
Abnormal    12
Normal       7
Name: count, dtype: int64


## Modality and Eye Distribution

In [ ]:
### Modality and eye distributions with breakdowns
if final_df.empty:
    print("Dataset is empty; run previous cells first.")
else:
    print("=== MODALITY OVERVIEW ===")
    mod_counts = final_df['modality'].fillna('Unknown').value_counts()
    mod_pct = (mod_counts / len(final_df) * 100).round(1)
    mod_table = (
        mod_counts.rename('count')
        .to_frame()
        .assign(percent=mod_pct)
    )
    print(mod_table.to_string())
    print("\nBy dataset (rows=dataset, cols=modality):")
    mod_by_ds = final_df.pivot_table(index='dataset_name', columns='modality', values='image_id', aggfunc='count', fill_value=0)
    print(mod_by_ds.to_string())
    
    print("\n=== EYE OVERVIEW ===")
    eye_counts = final_df['eye'].fillna('Unknown').value_counts()
    eye_pct = (eye_counts / len(final_df) * 100).round(1)
    eye_table = (
        eye_counts.rename('count')
        .to_frame()
        .assign(percent=eye_pct)
    )
    print(eye_table.to_string())
    print("\nEye by modality (rows=eye, cols=modality):")
    eye_by_mod = final_df.pivot_table(index=final_df['eye'].fillna('Unknown'), columns=final_df['modality'].fillna('Unknown'), values='image_id', aggfunc='count', fill_value=0)
    print(eye_by_mod.to_string())
    
    print("\nEye by dataset (rows=dataset, cols=eye):")
    eye_by_ds = final_df.pivot_table(index='dataset_name', columns=final_df['eye'].fillna('Unknown'), values='image_id', aggfunc='count', fill_value=0)
    eye_by_ds['unknown_%'] = (eye_by_ds.get('Unknown', 0) / eye_by_ds.sum(axis=1) * 100).round(1)
    print(eye_by_ds.sort_index().to_string())

## Summary and Next Steps

### ✅ Completed:
- Defined canonical harmonization schema with 16 standardized fields
- Implemented harmonization rules for diagnosis, modality, and laterality
- Built universal loader with auto-column detection
- Auto-detected datasets from INPUT (with demo fallback)
- Exported harmonized dataset to Parquet and CSV
- Verified data integrity and completeness

### 🔭 Next Steps:
1. **Integrate Real Kaggle Data**: Replace demo datasets with actual Kaggle API calls
2. **Expand Diagnosis Taxonomy**: Add more granular condition categories
3. **Extract Pixel Metadata**: Analyze image properties (resolution, aspect ratio)
4. **Implement Quality Checks**: Add validation for outliers and data anomalies
5. **Build Data Profiling Reports**: Generate per-dataset and cross-dataset summaries
6. **Add Duplicate Detection**: Use image hashing to identify similar images
7. **Create Train/Val/Test Splits**: Balance datasets across modalities and diagnoses

The harmonized dataset is ready for ML training and analysis!

In [ ]:
# Read back and verify the parquet file
print("\n=== VERIFICATION ===")
print("\nReading back Parquet file...")
loaded_df = pd.read_parquet(parquet_path)
print(f"✓ Loaded {len(loaded_df)} records from {parquet_path}")
print(f"  Shape: {loaded_df.shape}")
print(f"  Columns match: {list(loaded_df.columns) == list(final_df.columns)}")

In [ ]:
# Show sample of loaded data
print("\n=== SAMPLE OF LOADED DATA ===")
print(loaded_df.head(5)[['image_id', 'dataset_name', 'diagnosis_category', 'modality', 'eye']].to_string())

## Verify Exports

In [ ]:
# Ensure output directory exists
output_dir = Path('.') / 'output'
output_dir.mkdir(exist_ok=True)

# Export to Parquet (recommended for large datasets and efficient storage)
parquet_path = output_dir / 'harmonized.parquet'
final_df.to_parquet(parquet_path, index=False)
print(f"✓ Exported to Parquet: {parquet_path}")
print(f"  File size: {parquet_path.stat().st_size / 1024:.2f} KB")

# Export to CSV for easy inspection
csv_path = output_dir / 'harmonized.csv'
final_df.to_csv(csv_path, index=False)
print(f"\n✓ Exported to CSV: {csv_path}")
print(f"  File size: {csv_path.stat().st_size / 1024:.2f} KB")

## Export Harmonized Dataset

In [ ]:
# Patient metadata summary (when available)
print("\n=== PATIENT METADATA ===")
print(f"\nAge statistics (n={final_df['age'].notna().sum()}):")
print(final_df['age'].describe())

print(f"\nSex distribution:")
print(final_df['sex'].value_counts(dropna=False))

In [ ]:
# Data completeness
print("\n=== DATA COMPLETENESS ===")
completeness = (final_df.notna().sum() / len(final_df) * 100).sort_values(ascending=False)
print(completeness.to_string())

In [ ]:
# Modality and eye distribution
print("\n=== IMAGING CHARACTERISTICS ===")
print("\nModalities:")
print(final_df['modality'].value_counts(dropna=False))

print("\nEye distribution:")
print(final_df['eye'].value_counts(dropna=False))

## Structured Outputs
Combined summaries: overall stats, per-disease counts, healthy subset, per-dataset profiles, diagnosis × modality, laterality coverage, resolution stats, extra_json keys, and export readiness.

In [ ]:
# Structured summary outputs for quick review
print("\n=== ALL DATA ===")
print(f"Rows: {len(final_df)}")
print(f"Datasets: {final_df['dataset_name'].nunique()}")
print(f"Diagnosis categories: {final_df['diagnosis_category'].nunique()}")
print("\nModalities:")
modalities_table = final_df['modality'].value_counts(dropna=False).rename_axis('modality').reset_index(name='count')
print(modalities_table.to_string(index=False))
print("\n=== DATA PER DISEASE AREA ===")
disease_table = (
    final_df['diagnosis_category']
    .fillna('Unknown')
    .value_counts()
    .rename_axis('diagnosis_category')
    .reset_index(name='count')
)
print(disease_table.to_string(index=False))
print("\n=== HEALTHY DATA ===")
healthy_df = final_df[final_df['diagnosis_category'] == 'Normal']
if healthy_df.empty:
    print("No healthy records found.")
else:
    print(f"Healthy rows: {len(healthy_df)} across {healthy_df['dataset_name'].nunique()} datasets")
    print("Breakdown by dataset:")
    healthy_by_dataset = healthy_df['dataset_name'].value_counts().rename_axis('dataset_name').reset_index(name='count')
    print(healthy_by_dataset.to_string(index=False))
    print("\nSample records (up to 5):")
    sample_cols = ['image_id', 'dataset_name', 'image_path', 'eye', 'modality']
    print(healthy_df[sample_cols].head(5).to_string(index=False))

In [ ]:
# Extended summaries
from collections import Counter
import math
import json as _json
from pathlib import Path as _Path
import pandas as _pd

if final_df.empty:
    print("Dataset is empty; run upstream cells first.")
else:
    print("=== PER-DATASET PROFILE ===")
    def _pct(num, denom):
        return 0.0 if denom == 0 else round(num / denom * 100, 1)
    profiles = []
    for ds_name, grp in final_df.groupby('dataset_name'):
        profiles.append({
            'dataset_name': ds_name,
            'rows': len(grp),
            'patients': grp['patient_id'].nunique(dropna=True),
            'diagnoses': grp['diagnosis_category'].nunique(dropna=True),
            'modalities': grp['modality'].nunique(dropna=True),
            'missing_eye_%': _pct(grp['eye'].isna().sum(), len(grp)),
            'missing_age_%': _pct(grp['age'].isna().sum(), len(grp)),
            'missing_sex_%': _pct(grp['sex'].isna().sum(), len(grp)),
        })
    profile_df = _pd.DataFrame(profiles).sort_values('rows', ascending=False)
    print(profile_df.to_string(index=False))

    print("\n=== DIAGNOSIS x MODALITY (counts) ===")
    diag_mod_counts = _pd.crosstab(
        final_df['diagnosis_category'].fillna('Unknown'),
        final_df['modality'].fillna('Unknown')
    )
    print(diag_mod_counts.to_string())
    print("\nDIAGNOSIS x MODALITY (row %)" )
    diag_mod_pct = diag_mod_counts.div(diag_mod_counts.sum(axis=1), axis=0).fillna(0).round(3) * 100
    print(diag_mod_pct.to_string())

    print("\n=== LATERALITY COVERAGE BY DATASET ===")
    laterality = (
        final_df.assign(eye_cat=final_df['eye'].fillna('Unknown'))
        .pivot_table(index='dataset_name', columns='eye_cat', values='image_id', aggfunc='count', fill_value=0)
    )
    laterality['unknown_%'] = (laterality.get('Unknown', 0) / laterality.sum(axis=1) * 100).round(1)
    print(laterality.sort_index().to_string())

    print("\n=== RESOLUTION SUMMARY ===")
    if {'resolution_x', 'resolution_y'}.issubset(final_df.columns):
        res_df = final_df[['resolution_x', 'resolution_y']].dropna()
        if res_df.empty:
            print("No resolution data present.")
        else:
            res_df = res_df.assign(aspect_ratio=lambda d: d['resolution_x'] / d['resolution_y'])
            stats = {
                'rows_with_resolution': len(res_df),
                'mean_x': res_df['resolution_x'].mean(),
                'mean_y': res_df['resolution_y'].mean(),
                'median_ratio': res_df['aspect_ratio'].median(),
                'min_ratio': res_df['aspect_ratio'].min(),
                'max_ratio': res_df['aspect_ratio'].max(),
            }
            print(_pd.Series(stats).round(3).to_string())
            ratio_bins = _pd.cut(res_df['aspect_ratio'], bins=[0, 0.9, 1.1, math.inf], labels=['<0.9', '0.9-1.1', '>1.1'])
            print("\nAspect ratio buckets:")
            print(ratio_bins.value_counts().to_string())
    else:
        print("Resolution columns not available.")

    print("\n=== EXTRA_JSON KEY INVENTORY (top 20) ===")
    key_counter = Counter()
    key_to_ds = {}
    for _, row in final_df.iterrows():
        raw = row.get('extra_json')
        if not raw or raw in [None, 'None', '{}']:
            continue
        try:
            obj = _json.loads(raw) if isinstance(raw, str) else raw
            for k in obj.keys():
                key_counter[k] += 1
                key_to_ds.setdefault(k, set()).add(row.get('dataset_name'))
        except Exception:
            continue
    if not key_counter:
        print("No extra_json keys detected.")
    else:
        rows = []
        for k, count in key_counter.most_common(20):
            rows.append({'key': k, 'count': count, 'datasets': ', '.join(sorted(key_to_ds.get(k, [])))})
        print(_pd.DataFrame(rows).to_string(index=False))

    print("\n=== EXPORT READINESS ===")
    out_dir = _Path('.') / 'output'
    parquet_path = out_dir / 'harmonized.parquet'
    csv_path = out_dir / 'harmonized.csv'
    export_rows = []
    for path in [parquet_path, csv_path]:
        exists = path.exists()
        size_kb = path.stat().st_size / 1024 if exists else None
        export_rows.append({'file': str(path), 'exists': exists, 'size_kb': round(size_kb, 2) if size_kb else None})
    print(_pd.DataFrame(export_rows).to_string(index=False))